# Corazón v2 — Día 1: datos + U-Net (E1)

**Antes de correr (panel derecho de Kaggle → Settings):**
1. **Accelerator:** GPU T4 ×1.
2. **Internet:** ON (requiere cuenta con teléfono verificado). Se usa para bajar CheXmask (2 GB, PhysioNet) y los contornos de SCR (Zenodo).
3. **Add Input** (botón "+ Add Input"):
   - `nih-chest-xrays/data` → *NIH Chest X-rays* (112 120 radiografías, 42 GB; no se descarga, se monta).
   - `raddar/nodules-in-chest-xrays-jsrt` → radiografías **JSRT limpias** (la licencia de esta copia está POR CONFIRMAR en su página).
   - Tu dataset privado con el código: sube `corazon-v2-codigo.zip` como *New Dataset* con el nombre `corazon-v2-codigo`.
4. **Save Version → Save & Run All (Commit)**. Se puede cerrar el navegador y apagar la laptop. Al terminar, todo queda en *Output*.

**Duración estimada:** preparación ~20 min + 3 entrenamientos de U-Net ~20–30 min cada uno (estimación, no medida).

**Qué produce** (`/kaggle/working/`): `data/` (arrays listos para los días siguientes), `runs/E1*/` (pesos, `results.json`, `ict_unet.csv`), `results/E1_segmentacion.csv` y figuras de control.

In [ ]:
import glob, os, shutil, subprocess, sys, time
T0 = time.time()
W = "/kaggle/working"
REPO = f"{W}/repo"

# Código: desde el dataset privado subido a Kaggle (o desde GitHub si se define REPO_URL)
REPO_URL = ""   # p. ej. "https://github.com/<usuario>/corazon-v2"
if not os.path.exists(REPO):
    src = glob.glob("/kaggle/input/**/src/train_seg.py", recursive=True)
    if src:
        shutil.copytree(os.path.dirname(os.path.dirname(src[0])), REPO)
    elif REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
    else:
        raise SystemExit("No encuentro el código: sube corazon-v2-codigo.zip como dataset o define REPO_URL")
os.chdir(REPO)
print("código en", REPO, "->", sorted(os.listdir("src")))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])

In [ ]:
# Ubicar los datos montados (los nombres de carpeta de /kaggle/input pueden variar)
def buscar(patron):
    r = sorted(glob.glob(f"/kaggle/input/**/{patron}", recursive=True))
    return r[0] if r else ""

NIH_CSV = buscar("Data_Entry_2017*.csv")
TEST_LIST = buscar("test_list.txt")
NIH_ROOT = os.path.dirname(NIH_CSV)
jsrt = buscar("JPCLN001.*")
JSRT_DIR = os.path.dirname(jsrt)
print("NIH CSV  :", NIH_CSV)
print("test_list:", TEST_LIST or "NO (se hará partición aleatoria por paciente)")
print("JSRT     :", JSRT_DIR or "NO ENCONTRADO -> no habrá test dorado")
assert NIH_CSV, "Falta agregar el dataset nih-chest-xrays/data"

In [ ]:
# Descargas: CheXmask (máscaras débiles, CC BY 4.0) y contornos humanos de SCR (CC BY 4.0)
RAW = "/tmp/raw"; os.makedirs(RAW, exist_ok=True)
CHEXMASK = f"{RAW}/ChestX-Ray8.csv"
if not os.path.exists(CHEXMASK):
    subprocess.run(["wget", "-q", "-O", CHEXMASK,
        "https://physionet.org/files/chexmask-cxr-segmentation-data/1.0.0/OriginalResolution/ChestX-Ray8.csv"], check=True)
POINTS = f"{RAW}/points.zip"
if not os.path.exists(POINTS):
    subprocess.run(["wget", "-q", "-O", POINTS, "https://zenodo.org/records/7056076/files/points.zip?download=1"], check=True)
print({p: f"{os.path.getsize(p) / 1e6:.0f} MB" for p in (CHEXMASK, POINTS)}, f"{time.time() - T0:.0f}s")

In [ ]:
# Preparación: subconjunto (2 776 cardiomegalia + 6 000 sin cardiomegalia), partición por paciente,
# imágenes y máscaras a 256 px, ICT de referencia de CheXmask, test dorado de SCR
DATA = f"{W}/data"
cmd = [sys.executable, "-m", "src.prepare_data", "--nih-csv", NIH_CSV, "--nih-root", NIH_ROOT,
       "--chexmask", CHEXMASK, "--scr-points", POINTS, "--out", DATA, "--n-neg", "6000"]
if TEST_LIST: cmd += ["--test-list", TEST_LIST]
if JSRT_DIR: cmd += ["--scr-images", JSRT_DIR]
subprocess.run(cmd, check=True)
print(f"{time.time() - T0:.0f}s")

In [ ]:
# Controles antes de entrenar: cobertura, filtro RCA, proporción AP por split, y figuras de alineación
import numpy as np, pandas as pd, cv2
from src.prepare_data import unpack
sub = pd.read_csv(f"{DATA}/subset.csv")
print("imágenes:", len(sub), "| con máscara CheXmask:", int(sub.has_mask.sum()), "| RCA>=0.7:", int((sub.rca_mean >= 0.7).sum()))
print(sub.groupby(["split", "cardio"]).size().unstack())
print("proporción AP por split y clase:")
print(sub.assign(ap=sub.view.eq("AP")).groupby(["split", "cardio"]).ap.mean().unstack().round(3))
print("ICT CheXmask por clase (mediana):", sub.groupby("cardio").ICT_chexmask.median().round(3).to_dict())
os.makedirs(f"{W}/results", exist_ok=True)

def overlay(img, m):
    o = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for k, col in enumerate([(255, 0, 0), (0, 255, 0), (0, 0, 255)]):
        cs, _ = cv2.findContours(np.ascontiguousarray(m[k]), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(o, cs, -1, col, 1)
    return o

ni, nm = np.load(f"{DATA}/nih_images.npy", mmap_mode="r"), np.load(f"{DATA}/nih_masks.npy", mmap_mode="r")
k = sub.index[(sub.cardio == 1) & (sub.has_mask == 1)][:4]
cv2.imwrite(f"{W}/results/control_chexmask.png", np.concatenate([overlay(ni[i], unpack(nm[i])) for i in k], 1))
if os.path.exists(f"{DATA}/scr_images.npy"):
    si, sm = np.load(f"{DATA}/scr_images.npy"), np.load(f"{DATA}/scr_masks.npy")
    cv2.imwrite(f"{W}/results/control_scr.png", np.concatenate([overlay(si[i], unpack(sm[i])) for i in range(4)], 1))
print("figuras en results/: revisar que los contornos caigan sobre los órganos")

In [ ]:
from IPython.display import Image, display
for f in ("control_chexmask.png", "control_scr.png"):
    if os.path.exists(f"{W}/results/{f}"): display(Image(f"{W}/results/{f}"))

## E1 — Segmentación: etiquetas débiles vs. doradas
- **E1b** (principal): máscaras débiles filtradas por calidad (RCA ≥ 0,7).
- **E1a**: todas las máscaras débiles.
- **E1c**: E1b ajustada con 100 placas de contornos humanos (SCR).

La selección del mejor epoch usa la validación NIH; el test dorado (147 placas SCR) solo se mira al final.

In [ ]:
def entrenar(*extra):
    t = time.time()
    subprocess.run([sys.executable, "-m", "src.train_seg", "--data", DATA, "--out", f"{W}/runs", *extra], check=True)
    print(f"--- {extra[1]} terminado en {(time.time() - t) / 60:.1f} min (total {(time.time() - T0) / 60:.0f} min)")

entrenar("--run", "E1b_debil_rca07", "--masks", "rca07", "--epochs", "30")

In [ ]:
entrenar("--run", "E1a_debil_todas", "--masks", "all", "--epochs", "30")

In [ ]:
if os.path.exists(f"{DATA}/scr.csv"):
    entrenar("--run", "E1c_rca07_ft_scr", "--train-source", "scr", "--init", f"{W}/runs/E1b_debil_rca07/best.pt",
             "--epochs", "30", "--lr", "2e-4", "--bs", "8")

In [ ]:
# Tabla de E1 (con números propios) para el paper
import json
filas = []
for r in sorted(glob.glob(f"{W}/runs/E1*/results.json")):
    j = json.load(open(r))
    filas.append({"run": j["run"], "gpu": j["gpu"], "min_entrenamiento": round(j["train_time_min"], 1),
                  "mejor_epoch": j["best_val"]["epoch"], "val_dice_medio": round(j["best_val"]["val_dice_medio"], 4),
                  **{f"scr_{k}": (round(v, 4) if isinstance(v, float) else v) for k, v in j["scr_test_dorado"].items()}})
tabla = pd.DataFrame(filas)
tabla.to_csv(f"{W}/results/E1_segmentacion.csv", index=False)
display(tabla)
for d in glob.glob(f"{W}/runs/*/last.pt"): os.remove(d)   # los best.pt bastan; ahorra espacio en Output
print(f"tiempo total {(time.time() - T0) / 60:.0f} min")